In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
import json, pickle

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
ROOT = Path('../..').resolve()
ANALYSIS = ROOT / "analysis" / "affective_subspace_coverage"
ACTIVATION = ROOT / "activation" / "emotion_rewrites"
FIGURES = ROOT / "thesis" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── colour / style ─────────────────────────────────────────────────────────────
EMOTION_COLOURS = {
    "joy": "#F4C542",
    "trust": "#5BAD6F",
    "fear": "#7B5EA7",
    "surprise": "#F08030",
    "sadness": "#5B8DB8",
    "disgust": "#8B5E3C",
    "anger": "#D94040",
    "anticipation": "#E07840",
}
CANDIDATE_LAYERS = [10, 13, 16]

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

In [ ]:
def plot_layer_lineplot(save_path: Path) -> None:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df.sort_values("layer")

    metrics = [
        ("emo_probe_bacc_pc4",      "Emotion-category probe\nbalanced accuracy (4-D subspace)"),
        ("centroid_evr4",           r"Centroid compactness EVR$_4$"),
        ("mean_local_pc1_int_rho",  "Mean local PC1 –\nintensity Spearman ρ"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)

    for ax, (col, label) in zip(axes, metrics):
        ax.plot(df["layer"], df[col], marker="o", color="#2C5F8A", linewidth=1.8,
                markersize=6, zorder=3)

        # highlight layer 13
        val13 = float(df.loc[df["layer"] == 13, col].iloc[0])
        ax.axvline(13, color="#D94040", linewidth=0.8, linestyle="--", zorder=2,
                   label="Layer 13")
        ax.scatter([13], [val13], color="#D94040", zorder=4, s=50)

        ax.set_xlabel("Layer")
        ax.set_ylabel(label, labelpad=4)
        ax.set_xticks(df["layer"].tolist())
        ax.tick_params(axis="both", which="major", labelsize=9)

    axes[0].legend(fontsize=8, loc="lower right")
    fig.suptitle(
        "Layer-wise diagnostics for affective residual representations",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [ ]:
def load_activations():
    """Return H[N, E, I, L, D] and metadata.

    The .npy file may be a raw float32 binary without a numpy header (produced
    by a custom writer).  In that case np.load raises 'invalid load key'.  We
    fall back to np.memmap so the full 50 GB is never paged into RAM; only the
    slices actually accessed are read from disk.
    """
    info = json.loads((ACTIVATION / "emotion_intensity_residual_stream_info.json").read_text())
    npy_path = ACTIVATION / "emotion_intensity_residual_stream.npy"
    try:
        H = np.load(npy_path, allow_pickle=True)
        if isinstance(H, np.ndarray) and H.dtype == object:
            obj = H.item()
            if isinstance(obj, dict):
                H = max(obj.values(), key=lambda v: v.size if hasattr(v, "size") else 0)
            elif isinstance(obj, np.ndarray):
                H = obj
    except Exception:
        # File lacks a numpy header — memory-map as raw binary (no heap allocation)
        shape = tuple(info["shape"])
        dtype = np.dtype(info.get("dtype", "float32"))
        H = np.memmap(npy_path, dtype=dtype, mode="r", shape=shape)
    return H, info


In [ ]:
def plot_centroid_pca(save_path: Path) -> None:
    H, info = load_activations()
    layer_indices: list[int] = info["layer_indices"]   # e.g. [8,10,13,16,19,22]
    emotions: list[str]      = info["emotion_order"]
    # H shape: [N, E, I, L, D]

    target_layers = [10, 13, 16]
    layer_pos = {l: layer_indices.index(l) for l in target_layers}

    # mean over N (source texts) and I (intensities) → centroid per emotion per layer
    # H: [N, E, I, L, D]
    centroids = {}
    for l, lp in layer_pos.items():
        # mean over axis 0 (N) and axis 2 (I)
        c = H[:, :, :, lp, :].mean(axis=(0, 2))  # [E, D]
        centroids[l] = c

    # fit a shared PCA on all centroids concatenated
    all_c = np.concatenate([centroids[l] for l in target_layers], axis=0)  # [3E, D]
    pca = PCA(n_components=2)
    pca.fit(all_c)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

    for ax, l in zip(axes, target_layers):
        c_2d = pca.transform(centroids[l])  # [E, 2]
        for i, emo in enumerate(emotions):
            colour = EMOTION_COLOURS.get(emo, "#888888")
            ax.scatter(c_2d[i, 0], c_2d[i, 1], color=colour, s=70, zorder=3)
            ax.annotate(
                emo,
                (c_2d[i, 0], c_2d[i, 1]),
                textcoords="offset points",
                xytext=(5, 3),
                fontsize=7.5,
                color=colour,
            )
        ev1 = pca.explained_variance_ratio_[0] * 100
        ev2 = pca.explained_variance_ratio_[1] * 100
        ax.set_xlabel(f"PC1 ({ev1:.1f}%)", fontsize=9)
        ax.set_ylabel(f"PC2 ({ev2:.1f}%)", fontsize=9)
        ax.set_title(f"Layer {l}", fontsize=10, fontweight="bold")
        ax.axhline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.axvline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        "PCA of emotion centroids (shared basis) at candidate layers",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [ ]:
def generate_latex_table() -> str:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df[df["layer"].isin(CANDIDATE_LAYERS)].sort_values("layer")

    rows = []
    best = {
        "emo_probe_bacc_pc4": df["emo_probe_bacc_pc4"].max(),
        "centroid_evr4": df["centroid_evr4"].max(),
        "mean_local_pc1_int_rho": df["mean_local_pc1_int_rho"].max(),
    }

    for _, row in df.iterrows():
        layer = int(row["layer"])
        marker = r" \textbf{*}" if layer == 13 else ""

        def fmt(col, fmt_str):
            v = row[col]
            s = fmt_str.format(v)
            if abs(v - best[col]) < 1e-9:
                s = r"\textbf{" + s + "}"
            return s

        rows.append(
            f"  {layer}{marker} & "
            f"{fmt('emo_probe_bacc_pc4', '{:.4f}')} & "
            f"{fmt('centroid_evr4', '{:.4f}')} & "
            f"{fmt('mean_local_pc1_int_rho', '{:.4f}')} \\\\"
        )

    body = "\n".join(rows)
    table = r"""\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}
\end{table}"""
    return table

In [ ]:
plot_layer_lineplot(FIGURES / "layer_diagnostics_lineplot.pdf")

try:
    plot_centroid_pca(FIGURES / "layer_centroid_pca.pdf")
except Exception as e:
    print(f"[WARN] PCA scatter skipped: {e}")

tex = generate_latex_table()
print("\n── LaTeX table ────────────────────────────────────────────────")
print(tex)
out_path = FIGURES / "layer_diagnostics_table.tex"
out_path.write_text(tex)
print(f"\nSaved: {out_path}")

Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/layer_diagnostics_lineplot.pdf
Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/layer_centroid_pca.pdf

── LaTeX table ────────────────────────────────────────────────
\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
  10 & 0.6751 & 0.8876 & 0.0998 \\
  13 \textbf{*} & \textbf{0.7033} & 0.8920 & \textbf{0.1336} \\
  16 & 0.6704 & \textbf{0.8958} & 0.1297 \\
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}


In [ ]:

# ── CAA geometry paths ─────────────────────────────────────────────────────────
CAA_GEOMETRY = ROOT / "analysis" / "caa" / "geometry"


In [ ]:

def plot_intensity_consistency_heatmap(save_path: Path) -> None:
    """
    Heatmap of per-emotion, per-pair cosine similarity at layer 13.
    Colour range is clipped to [0.980, 1.000] to reveal fine differences.
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_intensity_consistency.csv")
    df13 = df[df["layer"] == 13].copy()

    emotions_order = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
    pairs_order    = ["low-medium", "low-high", "medium-high"]

    mat = (
        df13.pivot(index="emotion", columns="pair", values="cosine")
        .reindex(index=emotions_order, columns=pairs_order)
    )

    vmin, vmax = 0.980, 1.000

    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    im = ax.imshow(mat.values, aspect="auto", cmap="Blues",
                   vmin=vmin, vmax=vmax)

    ax.set_xticks(range(len(pairs_order)))
    ax.set_xticklabels(["low–medium", "low–high", "medium–high"], fontsize=9)
    ax.set_yticks(range(len(emotions_order)))
    ax.set_yticklabels([e.capitalize() for e in emotions_order], fontsize=9)

    # annotate cells
    for i in range(len(emotions_order)):
        for j in range(len(pairs_order)):
            val = mat.values[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=7.5,
                    color="white" if val < (vmin + (vmax - vmin) * 0.55) else "black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine similarity", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title("Intensity-pair cosine similarity per emotion (Layer 13)", fontsize=10)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")


In [ ]:

def plot_inter_emotion_cosine_heatmap(save_path: Path) -> None:
    """
    8×8 symmetric heatmap of inter-emotion cosine similarity (pooled CAA directions, Layer 13).
    Diagonal is masked. Colour scale spans [0, 1] to make the uniformly high values visually apparent.
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_cosine_similarity_L13.csv", index_col=0)
    emotions_order = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
    mat = df.reindex(index=emotions_order, columns=emotions_order).values.astype(float)

    # mask diagonal
    masked = np.ma.masked_where(np.eye(len(emotions_order), dtype=bool), mat)

    vmin, vmax = 0.0, 1.0

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad(color="#e8e8e8")  # diagonal colour
    im = ax.imshow(masked, aspect="equal", cmap=cmap, vmin=vmin, vmax=vmax)

    n = len(emotions_order)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    labels = [e.capitalize() for e in emotions_order]
    ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8.5)
    ax.set_yticklabels(labels, fontsize=8.5)

    # annotate off-diagonal only
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            val = mat[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=6.5,
                    color="white" if val > 0.6 else "black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine similarity", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title("Inter-emotion cosine similarity\nof pooled CAA directions (Layer 13)", fontsize=10)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")


In [ ]:

def generate_intensity_consistency_table() -> str:
    """
    LaTeX table summarising intensity-pair cosine similarity at Layer 13.
    Columns: Pair | Mean | Min | Max (across 8 emotions).
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_intensity_consistency.csv")
    df13 = df[df["layer"] == 13]

    pairs_order = ["low-medium", "low-high", "medium-high"]
    rows = []
    for pair in pairs_order:
        subset = df13[df13["pair"] == pair]["cosine"]
        mean_v = subset.mean()
        min_v  = subset.min()
        max_v  = subset.max()
        label  = pair.replace("-", "–")  # en-dash for typography
        rows.append(
            f"  {label} & {mean_v:.3f} & {min_v:.3f} & {max_v:.3f} \\\\"
        )

    body = "\n".join(rows)
    table = r"""\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Intensity pair} &
  \textbf{Mean cosine} &
  \textbf{Min cosine} &
  \textbf{Max cosine} \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\caption{Summary of intensity-pair cosine similarity at Layer 13, computed across
         all eight emotion categories. Even for the largest intensity gap
         (\textit{low–high}), direction consistency remains above 0.98,
         indicating that intensity modulation primarily scales rather than
         redirects the CAA vector.}
\label{tab:intensity_consistency_L13}
\end{table}"""
    return table


In [ ]:

# ── Generate CAA geometry figures & table ──────────────────────────────────────
plot_intensity_consistency_heatmap(FIGURES / "caa_intensity_consistency_heatmap_L13.pdf")
plot_inter_emotion_cosine_heatmap(FIGURES / "caa_inter_emotion_cosine_heatmap_L13.pdf")

tex_ic = generate_intensity_consistency_table()
print("\n── Intensity consistency LaTeX table ──────────────────────────────────")
print(tex_ic)
ic_path = FIGURES / "caa_intensity_consistency_table_L13.tex"
ic_path.write_text(tex_ic)
print(f"\nSaved: {ic_path}")


Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/caa_intensity_consistency_heatmap_L13.pdf
Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/caa_inter_emotion_cosine_heatmap_L13.pdf

── Intensity consistency LaTeX table ──────────────────────────────────
\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Intensity pair} &
  \textbf{Mean cosine} &
  \textbf{Min cosine} &
  \textbf{Max cosine} \\
\midrule
  low–medium & 0.996 & 0.995 & 0.998 \\
  low–high & 0.990 & 0.983 & 0.995 \\
  medium–high & 0.997 & 0.994 & 0.999 \\
\bottomrule
\end{tabular}
\caption{Summary of intensity-pair cosine similarity at Layer 13, computed across
         all eight emotion categories. Even for the largest intensity gap
         (\textit{low–high}), direction consistency remains above 0.98,
         indicating that intensity modulation primarily scales rather than
         redirects the CAA vector.}
\label{tab:intensity_consistency_L13}
\end{table}

Saved: /ho

## Steering Evaluation: Condition Comparison with 95% CI

Loads the per-sample LLM-judge scores (GPT-4o-mini) for the **n=376 test-split** decomposed-CAA
steering experiment and generates publication-quality figures with 95% confidence intervals
derived from the t-distribution (376 source texts x 8 target emotions = 3008 samples per condition;
376 per condition x emotion cell).

**Alignment note:** `generation_outputs.jsonl` begins with 800 legacy (old n=20) non-test rows;
the eval batch was built only from the test-split rows and indexed by `custom_id`. Scores are therefore
joined to generations **by custom_id**, not by a positional `zip`, which would be shifted by 800 rows.

All three evaluation metrics are scored on a 0-1 scale by the judge:
- **target_emotion_match**: does the generated text express the target emotion as its dominant affect?
- **meaning_preserved**: is the core propositional content of the source utterance retained?
- **emotionality**: is the text emotionally expressive at all (vs. affectively flat or degraded)?


In [ ]:
import scipy.stats as scipy_stats

STEER = ROOT / 'analysis' / 'caa' / 'steering_decomposition'
REWRITE_JSONL = ROOT / 'dataset' / 'emotion_rewrites' / 'emotion_rewrites.jsonl'

EMOTION_ORDER_STEER = [
    'joy', 'trust', 'fear', 'surprise',
    'sadness', 'disgust', 'anger', 'anticipation',
]

CONDITION_ORDER = ['none', 'g', 'resid', 'original_caa', 'g+resid']
CONDITION_LABELS = {
    'none':         'No steering',
    'g':            'Shared-only',
    'resid':        'Residual-only',
    'original_caa': 'Original CAA',
    'g+resid':      'Decomposed',
}
CONDITION_COLORS = {
    'none':         '#888888',
    'g':            '#4E9FC4',
    'resid':        '#C46E4E',
    'original_caa': '#8E70B8',
    'g+resid':      '#4CAF75',
}

# ── Align judge scores to generations by custom_id ─────────────────────────────
# generation_outputs.jsonl contains 800 *legacy* (old n=20) non-test rows at the
# front followed by the n=376 test-split rows.  The eval batch was built ONLY from
# test-split rows, in file order, with custom_id "exp1_{i:06d}" indexing into that
# filtered list.  A naive zip(gens, scores) is therefore shifted by the 800 legacy
# rows and mislabels every sample.  We instead rebuild the exact test-split row
# list and index the parsed scores by custom_id — the same logic the generation
# script uses to construct the batch.

# test-split source ids (first-seen order, mirroring extend_steering_to_n376.py)
_seen, TEST_SOURCE_IDS = set(), set()
for _line in REWRITE_JSONL.read_text().splitlines():
    if not _line.strip():
        continue
    _r = json.loads(_line)
    _sid = _r.get('source_id', '')
    if _sid and _sid not in _seen and _r.get('source_split') == 'test':
        _seen.add(_sid)
        TEST_SOURCE_IDS.add(_sid)

gens_all = [
    json.loads(l) for l in (STEER / 'generation_outputs.jsonl').read_text().splitlines()
    if l.strip()
]
# test-split rows in file order == the order the eval custom_ids index into
all_rows = [g for g in gens_all if g['source_id'] in TEST_SOURCE_IDS]

results_raw = [
    json.loads(l) for l in (STEER / 'generation_eval_results.jsonl').read_text().splitlines()
    if l.strip()
]
scores_by_idx = {
    int(r['custom_id'].split('_')[1]):
        json.loads(r['response']['body']['choices'][0]['message']['content'])
    for r in results_raw
}

SCORE_COLS = ['meaning_preserved', 'emotionality', 'target_emotion_match']
steer_df = pd.DataFrame([
    {
        'source_id':            all_rows[i]['source_id'],
        'steering_condition':   all_rows[i]['steering_condition'],
        'target_emotion':       all_rows[i]['target_emotion'],
        'meaning_preserved':    s['meaning_preserved'],
        'emotionality':         s['emotionality'],
        'target_emotion_match': s['target_emotion_match'],
    }
    for i, s in scores_by_idx.items()
])

n_per_cond = steer_df.groupby('steering_condition').size().min()
n_per_cell = steer_df.groupby(['steering_condition', 'target_emotion']).size().min()
print(f'Loaded {len(steer_df)} samples across {steer_df["steering_condition"].nunique()} conditions')
print(f'n per condition = {n_per_cond}   |   n per (condition x emotion) cell = {n_per_cell}')
print(steer_df.groupby('steering_condition')[SCORE_COLS].mean().round(3))


In [ ]:
def ci95(series: pd.Series) -> float:
    n = len(series)
    if n < 2:
        return float('nan')
    se = series.std(ddof=1) / n ** 0.5
    return float(scipy_stats.t.ppf(0.975, df=n - 1) * se)


def plot_condition_comparison(save_path: Path) -> None:
    metric_labels = {
        'target_emotion_match': 'Target-emotion match\n(LLM judge, 0-1)',
        'meaning_preserved':    'Semantic preservation\n(LLM judge, 0-1)',
        'emotionality':         'Emotionality\n(LLM judge, 0-1)',
    }
    metrics = list(metric_labels.keys())

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False)

    for ax, metric in zip(axes, metrics):
        means, errs, colors = [], [], []
        for cond in CONDITION_ORDER:
            vals = steer_df.loc[steer_df['steering_condition'] == cond, metric]
            means.append(vals.mean())
            errs.append(ci95(vals))
            colors.append(CONDITION_COLORS[cond])

        x = np.arange(len(CONDITION_ORDER))
        ax.bar(x, means, yerr=errs, capsize=5, color=colors,
               edgecolor='white', linewidth=0.6,
               error_kw={'elinewidth': 1.3, 'ecolor': '#333333', 'capthick': 1.3})
        ax.set_xticks(x)
        ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITION_ORDER],
                           rotation=25, ha='right', fontsize=8)
        ax.set_ylabel(metric_labels[metric], fontsize=9, labelpad=4)
        ax.tick_params(axis='y', labelsize=9)

        for xi, (m, e) in enumerate(zip(means, errs)):
            ax.text(xi, m + e + 0.006, f'{m:.3f}', ha='center', va='bottom', fontsize=7.5)

        ymin = min(means) - 4 * max(errs) - 0.02
        ymax = max(means) + 4 * max(errs) + 0.05
        ax.set_ylim(max(0.0, ymin), min(1.0, ymax))

    fig.suptitle(
        f'Steering intervention comparison  (n={n_per_cond} per condition; error bars = 95% CI)',
        fontsize=10, y=1.01,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_condition_comparison(FIGURES / 'steering_condition_comparison.pdf')


In [ ]:
def plot_per_emotion_target_match(save_path: Path) -> None:
    n_emo = len(EMOTION_ORDER_STEER)
    n_cond = len(CONDITION_ORDER)
    width = 0.14
    offsets = np.linspace(-(n_cond - 1) / 2 * width,
                          (n_cond - 1) / 2 * width, n_cond)

    fig, ax = plt.subplots(figsize=(13, 4.2))

    for ci, cond in enumerate(CONDITION_ORDER):
        means, errs = [], []
        for emo in EMOTION_ORDER_STEER:
            vals = steer_df.loc[
                (steer_df['steering_condition'] == cond) &
                (steer_df['target_emotion'] == emo),
                'target_emotion_match'
            ]
            means.append(vals.mean())
            errs.append(ci95(vals))

        x = np.arange(n_emo) + offsets[ci]
        ax.bar(x, means, width=width - 0.01,
               color=CONDITION_COLORS[cond], edgecolor='white', linewidth=0.4,
               label=CONDITION_LABELS[cond])
        ax.errorbar(x, means, yerr=errs, fmt='none',
                    ecolor='#333333', elinewidth=0.9, capsize=2.5, capthick=0.9)

    ax.set_xticks(np.arange(n_emo))
    ax.set_xticklabels([e.capitalize() for e in EMOTION_ORDER_STEER], fontsize=9)
    ax.set_ylabel('Target-emotion match (LLM judge, 0-1)', fontsize=9)
    ax.set_ylim(0, 0.85)
    ax.legend(fontsize=8, ncol=3, loc='upper right',
              framealpha=0.9, edgecolor='#cccccc')
    ax.set_title(
        f'Per-emotion target-match by intervention condition  (n={n_per_cell} per cell; error bars = 95% CI)',
        fontsize=9.5,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_per_emotion_target_match(FIGURES / 'steering_per_emotion_target_match.pdf')


In [ ]:
def plot_tradeoff_scatter(save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    for cond in CONDITION_ORDER:
        sub = steer_df[steer_df['steering_condition'] == cond]
        x_m = sub['meaning_preserved'].mean()
        y_m = sub['target_emotion_match'].mean()
        x_e = ci95(sub['meaning_preserved'])
        y_e = ci95(sub['target_emotion_match'])
        ax.errorbar(
            x_m, y_m, xerr=x_e, yerr=y_e,
            fmt='o', color=CONDITION_COLORS[cond], markersize=9,
            elinewidth=1.5, capsize=4, capthick=1.5,
            label=CONDITION_LABELS[cond], zorder=3,
        )
        ax.annotate(
            CONDITION_LABELS[cond],
            (x_m, y_m),
            textcoords='offset points', xytext=(8, 4),
            fontsize=7.5, color=CONDITION_COLORS[cond],
        )

    ax.set_xlabel('Semantic preservation (LLM judge, 0-1)', fontsize=9)
    ax.set_ylabel('Target-emotion match (LLM judge, 0-1)', fontsize=9)
    ax.set_title(
        f'Semantic preservation vs. target-emotion match\n(error bars = 95% CI, n={n_per_cond} per condition)',
        fontsize=9.5,
    )
    ax.grid(True, alpha=0.3, linewidth=0.5)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_tradeoff_scatter(FIGURES / 'steering_tradeoff_scatter.pdf')


In [ ]:
def plot_coefficient_sweep_proxy(save_path: Path) -> None:
    '''
    Proxy monotonicity check using the seed-robustness sweep.
    Shows output-length and repetition rate vs alpha_r at alpha_g=3.
    LLM-judge sweep scores were not available at reporting time; these proxy
    measures serve as an independent sanity check on coefficient directionality.
    Shaded band = +/- 1 SE over 8 seeds.
    '''
    ROB = ROOT / 'analysis' / 'caa' / 'seed_emotion_robustness'
    rob_path = ROB / 'seed_robustness_generations.jsonl'
    if not rob_path.exists():
        print(f'[SKIP] {rob_path} not found')
        return

    rob_df = pd.DataFrame([
        json.loads(l) for l in rob_path.read_text().splitlines() if l.strip()
    ])
    rob_df['alpha_g'] = pd.to_numeric(rob_df['alpha_g'], errors='coerce')
    rob_df['alpha_r'] = pd.to_numeric(rob_df['alpha_r'], errors='coerce')

    sweep = rob_df[
        (rob_df['alpha_g'] == 3.0) &
        (rob_df['steering_mode'] == 'last_token') &
        (rob_df['alpha_r'].isin([-5.0, -3.0, -1.5, 1.5, 3.0, 5.0]))
    ].copy()

    sweep['word_count'] = sweep['generated_text'].str.split().str.len().fillna(0)
    sweep['rep_rate'] = sweep['generated_text'].apply(
        lambda t: 1.0 - len(set(str(t).split())) / max(len(str(t).split()), 1)
        if isinstance(t, str) and t.strip() else float('nan')
    )

    alpha_r_vals = sorted(sweep['alpha_r'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

    for ax, (metric, ylabel) in zip(axes, [
        ('word_count', 'Mean output length (words)'),
        ('rep_rate',   'Mean repetition rate'),
    ]):
        for emo in EMOTION_ORDER_STEER:
            sub = sweep[sweep['target_emotion'] == emo]
            grp = sub.groupby('alpha_r')[metric]
            means = grp.mean().reindex(alpha_r_vals)
            sems  = (grp.std() / grp.count() ** 0.5).reindex(alpha_r_vals)
            c = EMOTION_COLOURS.get(emo, '#888888')
            ax.plot(alpha_r_vals, means.values, marker='o',
                    color=c, linewidth=1.5, markersize=5, label=emo.capitalize())
            ax.fill_between(
                alpha_r_vals,
                (means - sems).values, (means + sems).values,
                color=c, alpha=0.12,
            )

        ax.axvline(0, color='#bbbbbb', linewidth=0.8, linestyle='--')
        ax.set_xlabel(r'$\alpha_R$ (residual coeff., $\alpha_G=3$ fixed)', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xticks(alpha_r_vals)
        ax.tick_params(labelsize=9)

    axes[1].legend(fontsize=7.5, ncol=2, loc='upper left',
                   framealpha=0.9, edgecolor='#cccccc')
    fig.suptitle(
        r'Proxy monotonicity: output characteristics vs. $\alpha_R$ at $\alpha_G=3$'
        '  (8 seeds; shaded = +/- 1 SE)',
        fontsize=9.5, y=1.01,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_coefficient_sweep_proxy(FIGURES / 'steering_coefficient_sweep_proxy.pdf')


Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/steering_coefficient_sweep_proxy.pdf


In [ ]:
def generate_steering_condition_table() -> str:
    data = {}
    for cond in CONDITION_ORDER:
        sub = steer_df[steer_df['steering_condition'] == cond]
        data[cond] = {col: (sub[col].mean(), ci95(sub[col])) for col in SCORE_COLS}

    best = {col: max(data[c][col][0] for c in CONDITION_ORDER) for col in SCORE_COLS}

    display_names = {
        'none':         'No steering',
        'g':            r'Shared-only ($\alpha_G \mathbf{g}$)',
        'resid':        r'Residual-only ($\alpha_R \hat{\mathbf{r}}_e$)',
        'original_caa': r'Original CAA ($\alpha \mathbf{c}^{\mathrm{pool}}_e$)',
        'g+resid':      r'Decomposed ($\alpha_G \mathbf{g} + \alpha_R \hat{\mathbf{r}}_e$)',
    }

    def fmt_cell(col, cond):
        m, e = data[cond][col]
        s = f'{m:.3f} $\\pm$ {e:.3f}'
        if abs(m - best[col]) < 1e-9:
            s = '\\textbf{' + s + '}'
        return s

    rows = [
        f'  {display_names[cond]} & '
        f'{fmt_cell("target_emotion_match", cond)} & '
        f'{fmt_cell("meaning_preserved", cond)} & '
        f'{fmt_cell("emotionality", cond)} \\\\\n'
        for cond in CONDITION_ORDER
    ]

    body = '\n'.join(rows)
    table = (
        '\\begin{table}[H]\n'
        '\\centering\n'
        '\\small\n'
        '\\begin{tabular}{p{0.32\\linewidth}ccc}\n'
        '\\toprule\n'
        '\\textbf{Intervention} & '
        '\\textbf{Target-emotion match} & '
        '\\textbf{Semantic preservation} & '
        '\\textbf{Emotionality} \\\\\n'
        '\\midrule\n'
        + body +
        '\\bottomrule\n'
        '\\end{tabular}\n'
        f'\\caption{{Mean $\\pm$ 95\\% CI for each evaluation metric across five intervention'
        f' conditions ($n={n_per_cond}$ samples per condition). The 95\\% CI is computed from the'
        f' $t$-distribution with $n-1$ degrees of freedom. Bold marks the best value per column.}}\n'
        '\\label{tab:steering_results_ci}\n'
        '\\end{table}'
    )
    return table


tex_steer = generate_steering_condition_table()
print(tex_steer)
steer_table_path = FIGURES / 'steering_condition_table_ci.tex'
steer_table_path.write_text(tex_steer)
print(f'Saved: {steer_table_path}')


In [ ]:
# ── Paired condition comparisons (target-emotion match) ───────────────────────
# The five conditions are evaluated on the SAME (source_id, target_emotion) cells,
# so a paired test differences out the large between-emotion variance that inflates
# the marginal CIs in Table tab:steering_results_ci.  This is the correct inferential
# test for this within-item design.

# wide matrix: rows = (source_id, target_emotion), cols = condition -> target_emotion_match
wide = steer_df.pivot_table(
    index=['source_id', 'target_emotion'],
    columns='steering_condition',
    values='target_emotion_match',
)

def paired_stats(cond_a: str, cond_b: str):
    """Paired difference cond_a - cond_b over matched cells; returns dict."""
    d = (wide[cond_a] - wide[cond_b]).dropna()
    n = len(d)
    mean = d.mean()
    ci = scipy_stats.t.ppf(0.975, df=n - 1) * d.std(ddof=1) / n ** 0.5
    t = mean / (d.std(ddof=1) / n ** 0.5)
    p = 2 * scipy_stats.t.sf(abs(t), df=n - 1)
    # per-emotion agreement: how many of the 8 emotions move in the same (positive) direction
    per_emo = (
        steer_df[steer_df.steering_condition == cond_a]
        .groupby('target_emotion').target_emotion_match.mean()
        - steer_df[steer_df.steering_condition == cond_b]
        .groupby('target_emotion').target_emotion_match.mean()
    )
    n_pos = int((per_emo > 0).sum())
    return dict(a=cond_a, b=cond_b, n=n, mean=mean, ci=ci, t=t, p=p,
                n_pos=n_pos, n_emo=len(per_emo))

COMPARISONS = [
    ('resid', 'none'),
    ('resid', 'original_caa'),
    ('g+resid', 'original_caa'),
    ('original_caa', 'none'),
    ('g', 'none'),
]
paired_rows = [paired_stats(a, b) for a, b in COMPARISONS]
for r in paired_rows:
    agree = f"{r['n_pos']}/{r['n_emo']}" if (r['n_pos'] == r['n_emo'] or r['n_pos'] == 0) else '---'
    print(f"{CONDITION_LABELS[r['a']]:13s} - {CONDITION_LABELS[r['b']]:13s}: "
          f"d={r['mean']:+.3f} +/-{r['ci']:.3f}  t({r['n']-1})={r['t']:+.2f}  p={r['p']:.2g}  emotions+={agree}")


In [ ]:
def plot_paired_forest(save_path: Path) -> None:
    """
    Forest plot of paired mean differences (target-emotion match).
    Rows = comparisons; horizontal bars = 95% CI; colour = significant/not.
    """
    comparisons = [
        ('resid',        'none',         r'Residual-only $-$ No steering'),
        ('resid',        'original_caa', r'Residual-only $-$ Original CAA'),
        ('g+resid',      'original_caa', r'Decomposed $-$ Original CAA'),
        ('original_caa', 'none',         r'Original CAA $-$ No steering'),
        ('g',            'none',         r'Shared-only $-$ No steering'),
    ]

    rows = []
    for cond_a, cond_b, label in comparisons:
        r = paired_stats(cond_a, cond_b)
        rows.append(dict(label=label, mean=r['mean'], ci=r['ci'],
                         p=r['p'], n_pos=r['n_pos'], n_emo=r['n_emo']))

    # significant = p < 0.05
    SIG_POS  = '#2C7BB6'   # positive & significant
    SIG_NEG  = '#D7191C'   # negative & significant
    NSIG_COL = '#AAAAAA'   # not significant

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    y_positions = list(range(len(rows) - 1, -1, -1))   # top-to-bottom

    for yi, row in zip(y_positions, rows):
        sig = row['p'] < 0.05
        if sig and row['mean'] > 0:
            col = SIG_POS
        elif sig and row['mean'] < 0:
            col = SIG_NEG
        else:
            col = NSIG_COL

        ax.errorbar(row['mean'], yi, xerr=row['ci'],
                    fmt='o', color=col, markersize=7,
                    elinewidth=2, capsize=5, capthick=2, zorder=3)
        # p-value annotation
        if row['p'] < 0.001:
            p_str = f"p < 0.001"
        else:
            p_str = f"p = {row['p']:.3f}"
        ax.text(row['mean'] + row['ci'] + 0.001, yi, p_str,
                va='center', ha='left', fontsize=7.5, color=col)

    ax.axvline(0, color='#333333', linewidth=0.9, linestyle='--', zorder=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([r['label'] for r in rows], fontsize=9)
    ax.set_xlabel('Mean paired difference in target-emotion match (95% CI)', fontsize=9)
    ax.set_title(
        'Paired comparisons: target-emotion match\n'
        f'($n={n_per_cond}$ matched pairs per comparison)',
        fontsize=10,
    )

    # shaded region for "better than baseline"
    ax.axvspan(0, ax.get_xlim()[1] if ax.get_xlim()[1] > 0 else 0.04,
               alpha=0.04, color=SIG_POS, zorder=0)

    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=9)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


def plot_per_emotion_paired_delta(save_path: Path) -> None:
    """
    Per-emotion paired Δ for residual-only vs no-steering and vs original CAA.
    Shows the 8/8 consistency: every bar is positive.
    """
    emos = EMOTION_ORDER_STEER
    x = np.arange(len(emos))

    def emo_delta(cond_a, cond_b):
        means, cis = [], []
        for emo in emos:
            d = (
                steer_df.loc[(steer_df.steering_condition == cond_a) &
                             (steer_df.target_emotion == emo), 'target_emotion_match'].values
                - steer_df.loc[(steer_df.steering_condition == cond_b) &
                               (steer_df.target_emotion == emo), 'target_emotion_match'].values
            )
            # values are already matched within source_id for same emo; merge on source_id
            sub_a = steer_df[(steer_df.steering_condition == cond_a) &
                             (steer_df.target_emotion == emo)][['source_id','target_emotion_match']].set_index('source_id')
            sub_b = steer_df[(steer_df.steering_condition == cond_b) &
                             (steer_df.target_emotion == emo)][['source_id','target_emotion_match']].set_index('source_id')
            diffs = (sub_a - sub_b).dropna()['target_emotion_match']
            n = len(diffs); m = diffs.mean()
            ci_v = scipy_stats.t.ppf(0.975, df=n - 1) * diffs.std(ddof=1) / n ** 0.5
            means.append(m); cis.append(ci_v)
        return means, cis

    width = 0.35
    m1, ci1 = emo_delta('resid', 'none')
    m2, ci2 = emo_delta('resid', 'original_caa')

    fig, ax = plt.subplots(figsize=(11, 4.0))
    ax.bar(x - width / 2, m1, width=width - 0.03,
           color='#2C7BB6', label=r'Residual-only $-$ No steering', edgecolor='white', linewidth=0.5)
    ax.errorbar(x - width / 2, m1, yerr=ci1, fmt='none',
                ecolor='#1a4f7a', elinewidth=1.2, capsize=3.5, capthick=1.2)
    ax.bar(x + width / 2, m2, width=width - 0.03,
           color='#78C679', label=r'Residual-only $-$ Original CAA', edgecolor='white', linewidth=0.5)
    ax.errorbar(x + width / 2, m2, yerr=ci2, fmt='none',
                ecolor='#3a7a3a', elinewidth=1.2, capsize=3.5, capthick=1.2)

    ax.axhline(0, color='#333333', linewidth=0.9, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels([e.capitalize() for e in emos], fontsize=9)
    ax.set_ylabel('Mean paired $\\Delta$ in target-emotion match (95% CI)', fontsize=9)
    ax.set_title(
        'Per-emotion advantage of residual-only steering\n'
        f'($n={n_per_cell}$ matched pairs per bar; all bars positive = 8/8 emotions improved)',
        fontsize=9.5,
    )
    ax.legend(fontsize=8.5, loc='upper right', framealpha=0.9, edgecolor='#cccccc')
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_paired_forest(FIGURES / 'steering_paired_forest.pdf')
plot_per_emotion_paired_delta(FIGURES / 'steering_per_emotion_paired_delta.pdf')
